# 00 — AIDP environment check

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
import platform
raw = Path(config['bronze_raw_volume'])
landing = Path(config['bronze_landing_volume'])
print('Python:', platform.python_version())
print('Spark:', spark.version)
print('Working context:', Path.cwd())
print('Bronze raw volume:', raw, 'exists=', raw.is_dir())
print('Raw sessions:', {name: (raw / name).is_dir() for name in RAW_SESSIONS})
print('Bronze validated landing:', landing, 'exists=', landing.is_dir())
print('Catalog:', config['catalog'])
for schema_key in ('silver_schema', 'gold_historical_schema', 'gold_scenario_schema'):
    schema = config[schema_key]
    try:
        visible = spark.sql(f"SHOW SCHEMAS IN `{config['catalog']}`").filter(F.col('namespace') == schema).count() > 0
    except Exception as exc:
        visible = False
        print(f'Schema lookup issue for {schema}: {exc}')
    print(f'{schema_key}: {schema}; visible={visible}')
